---
**UNIVERSIDAD NACIONAL DE ASUNCIÓN**

**FACULTAD DE INGENIERÍA**

# **ROBÓTICA I**

TRABAJO PRÁCTICO FINAL 2026S2

## **"Robot móvil recogedor de frutas"**

---
###**Grupo [2]:**

---



*   Pablo Junior Otazu Lujan
*   Gabriela Belén Orrego Arzamendia
*   Héctor Dejesús Velázquez Ojeda
*   Mathias Ramón Aguilar Delvalle

**ENTREGA [6]:**  Demo control de motores en lazo cerrado

**Fecha:** 04/10/2026

**Versión:** [1.0]



## 1. Selección de Motores

Para el brazo recolector de limones se seleccionan motores con reducción y posibilidad de control de posición. La elección distingue el giro de la base, que orienta el brazo, de las articulaciones de elevación, que soportan el peso de los eslabones, la pinza y el limón.

---

### 1.1 Estimación de los Requisitos de Torque

El dimensionamiento debe considerar el peso del brazo, la pinza y una carga máxima de 200 g de limón. Se emplean las siguientes masas y longitudes aportadas por el equipo:

| Componente | Masa [g] | Longitud [mm] |
| :--- | ---: | ---: |
| Eslabón 1 ($m_1$, $L_1$) | 515,937 | 350,22 |
| Eslabón 2 sin pinza ($m_2$, $L_2$) | 362,640 | 329,52 |
| Pinza ($m_p$, $L_p$) | 322 | 190 |
| Limón: carga máxima ($m_f$) | 200 | — |

Los **684,64 g del segundo conjunto ya incluyen la pinza**; por ello, $m_2=684{,}64-322=362{,}64$ g. La masa móvil declarada es **1,2 kg sin fruta** y **1,4 kg con limón**. La suma de longitudes de los eslabones es **679,74 mm**; con la pinza alineada, el alcance geométrico hasta su extremo sería **869,74 mm**.

**Estimación estática conservadora.** Se toman $L_1$ y $L_2$ como distancias entre ejes, pendientes de confirmar, y se considera el brazo extendido horizontalmente. Como faltan los centros de masa y la pinza tiene distribución no homogénea, se concentra la masa de cada componente en su extremo más alejado del eje. Se coloca también el limón en el extremo de la pinza. Así se obtiene una cota del momento gravitacional mediante la suma de los pesos por sus brazos de palanca [R1]:

$$
\tau_{J_2,\mathrm{cota}}
=g\left[m_1L_1+m_2(L_1+L_2)+(m_p+m_f)(L_1+L_2+L_p)\right]
$$

$$
\tau_{J_3,\mathrm{cota}}
=g\left[m_2L_2+(m_p+m_f)(L_2+L_p)\right]
$$

Con las masas en kg, las longitudes en m y $g=9{,}81$ m/s²:

| Momento de carga equivalente | Cota con limón de 200 g [N·m] | Equivalente [kgf·cm] |
| :--- | ---: | ---: |
| Hombro ($J_2$) | **8,64** | **88,15** |
| Codo ($J_3$) | **3,83** | **39,08** |

Estas son cotas estáticas del modelo geométrico adoptado. El paralelogramo indicado utiliza bielas de aproximadamente **60 y 40 mm**, con relación geométrica **3:2**; esta relación no determina por sí sola el torque de los motores. Para completar su selección se debe verificar cómo transmite los esfuerzos y comparar el par requerido, incluyendo aceleración, fricción y pérdidas, con el **torque nominal continuo** de la variante instalada.

En la **base ($J_1$, eje vertical)**, el peso no genera torque alrededor del eje de giro. El requisito depende de $\tau_{J_1}=I_{\mathrm{eq}}\alpha+\tau_{\mathrm{fricción}}$; queda pendiente determinar la inercia del conjunto giratorio.


---

### 1.2 Comparación de Motores Disponibles

Se comparan los motores DC de la base y de las articulaciones con el servomotor DSSERVO RDS3218 utilizado en la pinza. Las características del servo se obtienen de su hoja de datos [R8].

| Parámetro | Motor 36GP-555 | Motor 5840-31ZY | Servo DSSERVO RDS3218 |
| :--- | :--- | :--- | :--- |
| **Tensión de alimentación** | 12 V | 12 V | 4,8–6,8 V |
| **Velocidad** | 160 rpm | 160 rpm | Sin carga: 0,19 s/60° a 5 V; 0,17 s/60° a 6,8 V |
| **Tipo de reductor** | Planetario (relación 50:1) | Tornillo sinfín (worm gear) | Reducción por engranajes, relación 373:1 |
| **Torque máximo** | 100 kgf·cm | 100 kg·cm | De bloqueo: 18,5 kgf·cm a 5 V; 21 kgf·cm a 6,8 V |
| **Encoder** | Hall integrado (16 PPR en eje motor) | Óptico externo 38S6G5-B-G24N (1000 PPR en salida) | -- |
| **Cuentas por vuelta ($\times 4$)** | 3200 cuentas/vuelta | 4000 cuentas/vuelta | -- |
| **Autobloqueo mecánico** | No | Sí | -- |


---

### 1.3 Justificación de la Elección por Articulación

* **Base ($J_1$) — Motor 36GP-555:**
  Como la base gira únicamente en el plano horizontal $XY$, la gravedad no genera par resistente sobre este eje. La carga principal es vencer el par de inercia al acelerar o frenar. La reductora planetaria provee un giro continuo y suave sin el juego excesivo del sinfín, y el encoder Hall integrado simplifica el montaje físico.

  <p align="center">
  <img src="https://lh3.googleusercontent.com/d/1Pr8gyXGx4nPgvVkYJcDFzcoqgBlERl8A" width="600" alt="Conexiones del motor base" />
  <br>
  <b>Fig. 1.</b> Fotografía del Motor 36GP-555).
</p>

* **Hombro ($J_2$) y Codo ($J_3$) — Motor 5840-31ZY:**
  Ambas articulaciones soportan el peso en voladizo del brazo y del limón. Se seleccionó este motor por su elevado torque de salida y su característica de **autobloqueo**: la geometría del tornillo sinfín evita que el brazo caiga por gravedad cuando los motores están desenergizados. Como contraparte, la fricción estática del sinfín genera una zona muerta amplia que el controlador PID deberá compensar.

  <p align="center">
  <img src="https://lh3.googleusercontent.com/d/1skDKAEs6KgkNk7QWiOdMU7SeGibIM-iu" width="600" alt="Conexiones del motor base" />
  <br>
  <b>Fig. 2.</b> Fotografía del motor 5840-31ZY.
</p>

* **Estandarización en $J_2$ y $J_3$:**
  Usar el mismo modelo de motor en ambas articulaciones simplifica el diseño mecánico, la programación y la organización de los repuestos. La diferencia en la carga física entre hombro y codo se corregirá ajustando las ganancias del controlador ($K_p, K_i, K_d$) de forma independiente para cada motor.


### 1.3 Justificación de la Elección del Servo

La selección de los servos RDS3218 por su control de posición mediante PWM, su alto torque y su masa de 60 g, que reduce la carga añadida al extremo del brazo. Un servo ajusta la apertura y el cierre para sujetar el limón; el otro gira la pinza para orientar el agarre. La fuerza de cierre deberá validarse según la geometría del mecanismo y la carga.

<p align="center">
  <img src="https://lh3.googleusercontent.com/d/11eEpPnE30q2pAb1sC0j3td0bn-qoPLTZ" width="600" alt="Servo motor" />
  <br>
  <b>Fig. 3.a.</b> Fotografía del Servo RDS2319.
</p>



---
# 2. Drivers



Para el manejo de los actuadores se seleccionaron dos arquitecturas de potencia distintas, adaptadas a las exigencias de corriente y a la estrategia de control de cada articulación.

---

### 2.1 Motor de la Base ($J_1$): Driver IBT-2 (BTS7960)

El driver IBT-2 está integrado por dos medios puentes H basados en el C.I. BTS7960. Este módulo soporta corrientes de hasta 43 A, capacidad sobredimensionada respecto a la demanda del motor 36GP-555, lo que garantiza una operación segura y libre de sobrecalentamiento.

<p align="center">
  <img src="https://lh3.googleusercontent.com/d/1emxgWWbqDXQjHAXYHlnz8Uj_UmHrhHtg" width="600" alt="Conexiones del motor base" />
  <br>
  <b>Fig. 3.</b> Fotografía del Driver IBT-2 (BTS7960).
</p>

**Estrategia de control y configuración:**
* **Señales PWM independientes:** Se utilizan dos líneas PWM (`RPWM` y `LPWM`) para comandar el sentido de giro directo e inverso respectivamente.
* **Frecuencia de conmutación:** Se fijó una frecuencia de PWM de **20 kHz**. Este valor se ubica por encima del rango audible para evitar ruidos molestos y asegura una corriente más continua en el devanado.
* **Compatibilidad de niveles:** El pin `VCC` del driver se alimenta con 3,3 V para compatibilizar las entradas lógicas directamente con los niveles del ESP32-S3.

<div align="center">
  <img src="https://lh3.googleusercontent.com/d/1LNEMibOYLk5iFPIgzS0DTYiAAM5_VVKP" alt="Conexiones IBT-2" width="55%"/>
  <p><b>Fig. 4.</b> Esquema de conexiones para la base ($J_1$): Motor 36GP-555, IBT-2 y ESP32-S3.</p>
</div>

| Señal | Pin ESP32-S3 | Función |
| :--- | :--- | :--- |
| **RPWM** | GPIO4 | PWM giro sentido horario |
| **LPWM** | GPIO5 | PWM giro sentido antihorario |
| **R_EN** | GPIO6 | Habilitación canal derecho |
| **L_EN** | GPIO7 | Habilitación canal izquierdo |
| **VCC (Lógica)** | 3.3V | Referencia lógica |
| **Encoder Canal A** | GPIO15 | Lectura de cuadratura |
| **Encoder Canal B** | GPIO16 | Lectura de cuadratura |
| **Alimentación Encoder** | 3.3V / GND | Alimentación del sensor Hall |

---

### 2.2 Motores de las Articulaciones ($J_2$ y $J_3$): Driver L298N

Para las articulaciones de elevación y extensión se utiliza el módulo L298N, el cual contiene un puente H doble basado en transistores bipolares (BJT). Un único módulo permite controlar de forma independiente ambos motores 5840-31ZY.

<p align="center">
  <img src="https://lh3.googleusercontent.com/d/1cZd0fuUJRjgpL6PIryoVDdBh0riffkX0" width="600" alt="Conexiones del motor base" />
  <br>
  <b>Fig. 5.</b> Fotografía del Driver L298N.
</p>

**Estrategia de control y observaciones:**
* **Modo de operación (Freno dinámico):** Se retiró el jumper de habilitación (`ENA`) para conectarlo a un GPIO y mantenerlo en nivel alto dinámico. El control de velocidad y sentido se realiza aplicando PWM a 1 kHz sobre las entradas `IN1` e `IN2`. Cuando ambas entradas están en bajo y `ENA` en alto, el motor entra en modo de freno dinámico, ayudando a estabilizar la posición.
* **Caída de tensión interna:** Debido a la caída de tensión en los BJT del L298N (~2,4 V), alimentando el driver con 12 V la tensión efectiva que recibe el motor a PWM máximo es de aproximadamente **9,6 V** (medido experimentalmente en la Entrega 5).

<div align="center">
  <img src="https://lh3.googleusercontent.com/d/1bzLj3Rn3PKOfB4V7OrrVnqVOa7pddPw3" alt="Conexiones L298N" width="55%"/>
  <p><b>Figura 6.</b> Esquema de conexiones para la articulación ($J_2$): Motor 5840-31ZY, L298N y ESP32-WROOM-32D.</p>
</div>


| Señal | Pin ESP32-WROOM-32D | Función |
| :--- | :--- | :--- |
| **ENA** | GPIO25 | Habilitación del puente H (Freno/Enable) |
| **IN1** | GPIO26 | Entrada de control 1 (PWM) |
| **IN2** | GPIO27 | Entrada de control 2 (PWM) |
| **Encoder Canal A** | GPIO32 | Entrada de fase A (Cable blanco) |
| **Encoder Canal B** | GPIO33 | Entrada de fase B (Cable verde) |
| **Alimentación Encoder** | 5V (VIN) / GND | Alimentación del encoder óptico (Cable rojo) |

---

### 2.3 Consideraciones Eléctricas y de Seguridad

1. **Acondicionamiento del Encoder Óptico:** El encoder 38S6G5 cuenta con salidas NPN de colector abierto. Para fijar la lógica de 3,3 V requerida por los GPIO del ESP32 sin riesgos de sobretensión, se configuraron las resistencias de *pull-up* internas del microcontrolador.
2. **Referencia Común de Masa:** Se interconectaron las masas (GND) de la fuente de alimentación, los módulos drivers y las placas ESP32 para asegurar un plano de referencia unificado y evitar errores de lectura o disparos falsos.
3. **Protección de Corriente en Fuente:** Las pruebas se realizan alimentando los circuitos con una fuente regulada a 12 V, fijando un límite de corriente de 3 A para la prueba de la base ($J_1$) y de 2 A para las articulaciones ($J_2$/$J_3$), evitando daños ante eventuales bloqueos mecánicos.

---
### **3 - Control de lazo abierto**

En la Entrega 5 se caracterizaron ambos motores en lazo abierto aplicando un PWM conocido y midiendo con el encoder la respuesta del eje, sin realimentación. Los ensayos fueron:

- **Barrido de PWM**, para obtener la curva velocidad-PWM, la ganancia y la zona muerta.
- **Escalón de PWM**, para obtener la constante de tiempo.
- **Perfil en escalera** ($\pm 20\dots 100\,\%$), con ajuste de un modelo de primer orden a cada tramo mediante `scipy.optimize.curve_fit`.
- **Rampa con y sin carga** (motor base), para evaluar el efecto de la carga.

La adquisición se realizó con un período de muestreo de 10 ms y lectura del encoder por PCNT.

#### Resultados resumidos

| Parámetro | Motor base (36GP-555 + IBT-2) | Motor articulación (5840-31ZY + L298N) |
| :--- | :--- | :--- |
| Ganancia estática $K$ | 1,58 rpm/% | 1,33 rpm/% |
| Constante de tiempo $\tau$ | 65 ms | 59 ms |
| Rango lineal | 20 a 85 % | 40 a 100 % |
| Zona muerta (arranque) | 16 % (vacío), 25 % (con carga) | 20 a 40 % |
| Velocidad máxima | ~145 rpm | ~140 rpm |
| Simetría y repetibilidad | error < 2,5 rpm / ±1,5 rpm | error < 5 rpm / ±5 rpm |

<p align="center">
  <img src="https://raw.githubusercontent.com/HecVelaz/pruebas-esp32/main/06-motor-36gp555/resultados/barrido_20260925_172703.png" width="650"><br>
  <b>Fig. 7a.</b>  Barrido de PWM del motor base (36GP-555 + IBT-2), en los dos sentidos: la velocidad crece en forma lineal con el PWM (~1,6 rpm/%) desde ~11 % y es simétrica. Arriba de ~85 % la curva salta y se aplana en ~145 rpm.
</p>

<p align="center">
  <img src="https://raw.githubusercontent.com/HecVelaz/pruebas-esp32/main/06-motor-36gp555/resultados/perfil_20260925_175338.png" width="750"><br>
  <b>Fig. 7b.</b>  Perfil en escalera del motor base (±20 a 100 % de PWM, 5 s por escalón): posición, PWM aplicado, respuesta en cada nivel y velocidad. Cada escalón llega a su velocidad en unos 65 ms (τ) y la respuesta es simétrica.
</p>

<p align="center">
  <img src="https://raw.githubusercontent.com/HecVelaz/pruebas-esp32/main/08-motor-5840-l298n/resultados/perfil_20260925_215823.png" width="750"><br>
  <b>Fig. 7c.</b>  Perfil en escalera del motor de articulación (5840-31ZY + L298N): con ±20 % no se mueve (zona muerta) y desde 40 % la velocidad crece casi en forma lineal hasta ~140 rpm a 100 %.
</p>

#### Limitaciones de esta caracterización

- Los ensayos se hicieron con el eje libre (en el motor base, también con una carga de prueba). Con el brazo armado, la gravedad y la inercia de los eslabones cambian la zona muerta, $K$ y $\tau$. Por eso se repitió la caracterización con el brazo montado: en J3 (codo) se identificó un modelo nuevo (sección 5), y en J1 (base) el modelo con el eje libre alcanzó para predecir la respuesta.
- **Zona muerta del 5840 con el brazo montado:** se midió con barridos lentos a velocidad constante (4 °/s), en los dos sentidos y en todo el rango de ángulos, con el driver IBT-2 a 1 kHz. En **J3 (codo)**, el PWM para subir crece con el ángulo por el peso del antebrazo: $26{,}1 + 1{,}42\,\theta$ % (dispersión ±4,9 %); para bajar alcanza con −5,7 % en todo el rango. En **J2 (hombro, con un resorte que compensa parte del peso)**, para subir hacen falta ~59 % a −10 °, ~55 % a 0 ° y ~48 % a +20 °, y para bajar, −9 a −10 % en todo el rango. Así, la zona muerta dejó de ser un rango amplio (20 a 40 %) y pasó a ser un valor conocido según el ángulo y el sentido, que el firmware compensa.
- **Por encima del 85 % de PWM** la velocidad del motor base deja de crecer de forma lineal. La causa más probable es el **driver**: el BTS7960 del IBT-2 no alcanza a conmutar bien a 20 kHz. En J3, con el mismo driver a 20 kHz, el motor recibía menos tensión de la esperada (0,6 V en lugar de 1,8 V con 15 % de PWM), y al bajar a 1 kHz rendía 2 a 3 veces más. En el motor base no se usa más de 40 % de PWM, así que no afecta al control. Para confirmarlo, falta repetir el barrido del motor base a 1 kHz o medir la tensión de la fuente durante el barrido.

---
### **4 - Sensores**

Cada articulación utiliza un encoder incremental en cuadratura (canales A y B), que permite contar pulsos y determinar el sentido de giro. La lectura se realiza en el ESP32 con el contador de hardware **PCNT**, en modo **x4** (se cuentan los flancos de subida y bajada de ambos canales) y con un filtro de ruido de 10 µs.

#### Resolución

| Articulación | Encoder | Cálculo | Cuentas por vuelta de salida | Resolución |
| :--- | :--- | :--- | :--- | :--- |
| J1 | Hall, 16 PPR en el eje del motor | $16 \times 50 \times 4$ | 3200 | 0,1125 ° |
| J2 y J3 | Óptico 38S6G5, 1000 PPR en el eje de salida | $1000 \times 4$ | 4000 | 0,09 ° |

<p align="center">
  <img src="https://lh3.googleusercontent.com/d/1DtKXAiR-xmWliY_SiW1RXKpvLsRmcdRu" width="600" alt="Conexiones del motor base" />
  <br>
  <b>Fig. 8.</b> Fotografía del encoder incremental del motor de la base J1.
</p>

<p align="center">
  <img src="https://lh3.googleusercontent.com/d/18IYumGK25AwwS9MPfuJedp2uF77hZukq" width="600" alt="Conexiones del motor base" />
  <br>
  <b>Fig. 9.</b> Fotografía del encoder óptico para las articulaciones J2 y J3.
</p>



#### Conversiones

$$\theta\,[^\circ] = \frac{N \cdot 360}{CPR}, \qquad \omega\,[\text{rpm}] = \frac{\Delta N}{CPR \cdot T_s} \cdot 60$$

donde $N$ es el conteo acumulado, $CPR$ las cuentas por vuelta de salida y $T_s = 10$ ms. Además, $1\ \text{rpm} = 6\ ^\circ/\text{s}$.



#### Observaciones

- El encoder del 36GP-555 está antes de la reductora, por lo que no detecta la holgura de los engranajes. El del 5840 mide el eje de salida, a través de un acople flexible.
- Los encoders incrementales pierden la referencia al apagarse. En esta demo el cero se fija manualmente con el eje en una posición conocida. En el brazo se agregarán finales de carrera para hacer el homing al encender.


---
### **5 - Control de lazo cerrado: modelo de motor**

Para un motor de corriente continua controlado por tensión de armadura, la función de transferencia entre la posición angular y la tensión es

$$
G(s) = \frac{\theta(s)}{V(s)} = \frac{K_t}{s\left[(Js + b)(Ls + R) + K_t K_b\right]}
$$

**Donde:**

- $K_t$: constante de torque [N·m/A]; $K_b$: constante de fuerza contraelectromotriz [V·s/rad].
- $J$: inercia [kg·m²]; $b$: fricción viscosa [N·m·s/rad].
- $L$: inductancia de armadura [H]; $R$: resistencia de armadura [Ω].

Con la inductancia despreciable ($L \approx 0$) se obtiene el modelo de segundo orden

$$
G(s) = \frac{K_m}{s(\tau_m s + 1)}, \qquad K_m = \frac{K_t}{R\,b + K_t K_b}, \qquad \tau_m = \frac{R\,J}{R\,b + K_t K_b}
$$

Como no se dispone de todos los parámetros físicos, el modelo se identificó experimentalmente (sección 3). La velocidad responde como un sistema de primer orden,

$$
G_\omega(s) = \frac{\Omega(s)}{U(s)} = \frac{K}{\tau s + 1}
$$

y la posición es su integral:

$$
G_\theta(s) = \frac{\theta(s)}{U(s)} = \frac{K_\theta}{s(\tau s + 1)}
$$

con la entrada $U$ en **% de PWM** y $K_\theta$ expresada en °/s por % de PWM ($K_\theta = 6\,K$ con $K$ en rpm/%).

#### Modelos identificados

| Articulación | $K$ [rpm/%] | $K_\theta$ [°/s por %] | $\tau$ [s] | Modelo de posición |
| :--- | :--- | :--- | :--- | :--- |
| J1 (36GP-555), eje libre | 1,58 | 9,5 | 0,065 | $\dfrac{9{,}5}{s(0{,}065\,s + 1)}$ |
| J1 en la base, montado | 0,32 | 1,90 | 0,065 | $\dfrac{1{,}90}{s(0{,}065\,s + 1)}$ |
| J2 y J3 (5840-31ZY), eje libre | 1,33 | 8,0 | 0,059 | $\dfrac{8{,}0}{s(0{,}059\,s + 1)}$ |
| J3 en el codo, brazo montado | 0,27 | 1,62 | 0,053 | $\dfrac{1{,}62}{s(0{,}053\,s + 1)}$ |

Los modelos "eje libre" se midieron con el motor solo. El de J1 en la base sale de dividir por la correa (5:1). El de J3 en el codo se **identificó con el brazo montado**, con los datos guardados de 8 escalones de velocidad y 2 barridos de carga: se le aplicó al modelo el mismo PWM que se aplicó al motor y se ajustaron $K$ y $\tau$ hasta que la velocidad simulada coincidiera con la medida.

#### No linealidades y limitaciones

- **Zona muerta:** el motor no se mueve por debajo de cierto PWM: 17 % en J1 (en la base) y 20 a 40 % en J2 con el eje libre. En J3 con el brazo montado depende del ángulo por el peso del antebrazo: subiendo $u_c = 23{,}6 + 1{,}42\,\theta$ % y bajando 3,2 %. Un controlador proporcional dejaría un error final, por lo que el diseño incluye compensación de zona muerta (en J3, también del peso según el ángulo) y acción integral.
- **Saturación:** el PWM está limitado a $\pm 100\,\%$.
- **Holgura y fricción** del reductor, en especial del tornillo sin fin.
- **Carga:** en J2 y J3 el torque gravitatorio depende del ángulo. J3 ya se re-identificó con el brazo montado (fila "J3 en el codo"); J2 todavía falta.

---
### **6 - Simulación**

La simulación se realiza en **Python** (`numpy`, `scipy` y `matplotlib`) sobre los modelos de la sección 5, con los mismos períodos de muestreo que el firmware: $T_s = 10$ ms para el lazo de velocidad y $20$ ms para el de posición. El simulador usa el mismo controlador que corre en la placa (mismas ganancias, límites y compensaciones), así que la simulación se puede comparar directamente con las pruebas.

#### Especificaciones de diseño

| Especificación | Valor | J1 (base), placa | J3 (codo), placa |
| :--- | :--- | :--- | :--- |
| Sobrepico de posición | 0 | 0 | 0 |
| Tiempo de llegada (a menos de 0,5 °) | ≤ 2 s | 1,71 a 1,77 s (±20 °) | 0,96 a 1,92 s (±10 °) |
| Error en régimen permanente | ≤ 0,5 ° | ≤ 0,30 ° | ≤ 0,29 ° |

El lazo de velocidad se diseñó para un tiempo de establecimiento de 0,25 s sin sobrepico.

#### Controlador

Se utiliza un control **en cascada**: un lazo de posición (P) pide una velocidad y un lazo de velocidad (PI) calcula el PWM.

<p align="center">
  <img src="https://raw.githubusercontent.com/HecVelaz/pruebas-esp32/main/10-control-lazo-cerrado/simulaciones/diagrama_control.png" width="750"><br>
  <b>Fig. 10.</b>  Diagrama del control en cascada. El lazo de posición (P) pide una velocidad y el lazo de velocidad (PI) calcula el PWM del motor; el encoder mide el ángulo y, con él, la velocidad.
</p>

**Lazo de velocidad (PI, cada 10 ms):**

$$
u[k] = u_{ff} + K_p\, e[k] + K_i\, T_s \sum_{j=0}^{k} e[j], \qquad e[k] = \omega_{ref}[k] - \omega[k]
$$

**Lazo de posición (P, cada 20 ms):**

$$
\omega_{ref} = K_{pp}\,(\theta_{ref} - \theta)
$$

con las siguientes características:

- **Compensación de zona muerta ($u_{ff}$):** se suma el PWM mínimo para que el motor se mueva, en el sentido pedido. En J3 también se suma la fuerza para levantar el peso del antebrazo según el ángulo: $u_{ff} = 24 + 1{,}4\,\theta$ % subiendo y 9 % bajando.
- **Diseño por cancelación de polo:** con $K_p / K_i = \tau$, el lazo de velocidad queda $\dfrac{1}{\tau_{lc}s + 1}$ con $\tau_{lc} = \dfrac{\tau}{K K_p}$.
- **Sin sobrepico de posición** si $K_{pp} \le \dfrac{1}{4\,\tau_v}$, donde $\tau_v$ es lo que tarda en responder el lazo de velocidad.
- **Perfil de movimiento:** la velocidad pedida se limita a $v_{max}$ y su cambio a 60 °/s², para no golpear la mecánica.
- **Tolerancia de llegada** de ±0,3 °: dentro de ella el motor frena (y no vuelve a moverse hasta que se aleja más de 0,6 °), para que no zumbe alrededor del objetivo.
- **Anti-windup:** la integral no crece mientras el PWM está saturado.

#### Ganancias

| | $K_p$ | $K_i$ | $K_{pp}$ | $v_{max}$ | Límite sin sobrepico |
| :--- | :--- | :--- | :--- | :--- | :--- |
| J1 (base) | 0,549 | 4 | 2 | 30 °/s | $K_{pp} \le 2{,}78$ |
| J3 (codo) | 0,35 | 5 | 2 | 25 °/s | $K_{pp} \le 2{,}08$ |

Las ganancias de diseño eran más altas ($K_i = 8{,}44$ en J1; $K_p = 0{,}52$ y $K_i = 9{,}9$ en J3). En la placa se bajaron: en J1 por el juego de la correa, que daba 18 % de sobrepico de velocidad, y en J3 porque la fricción del tornillo sin fin lo hacía oscilar.

#### Efectos incluidos en el simulador

- Zona muerta (en J3, distinta subiendo y bajando, y dependiente del ángulo por el peso) y fricción para arrancar desde quieto.
- Saturación de PWM.
- Cuantización del encoder (0,0225 ° en la base de J1 y 0,029 ° en el codo de J3) y velocidad medida con la diferencia de cuentas en 40 ms, igual que el firmware.
- Muestreo a 100 Hz (velocidad) y 50 Hz (posición), y el perfil de movimiento con la tolerancia de llegada.

**No incluidos:** la holgura (juego) de la correa y de los engranajes, el "traba-suelta" del tornillo sin fin al bajar despacio y el endurecimiento del mecanismo del codo cerca de su posición más baja. Son la causa de las diferencias con la placa.

#### Escenarios

1. Escalones de velocidad: ±20 °/s en J1 y ±10 °/s en J3.
2. Escalones de posición: ±20 ° en J1, y ±10 ° y +20 ° en J3.
3. Las mismas secuencias de escalones de las pruebas en la placa, para comparar la simulación con la medición.

Para cada escenario se grafican la referencia, la posición, la velocidad y el PWM, y se calculan el sobrepico, el tiempo de llegada y el error final.

#### Resultados de la simulación

**J1 (base):** escalones de posición de 20 ° y 5 °. Llega en 1,64 s sin sobrepico.

<p align="center">
  <img src="https://raw.githubusercontent.com/HecVelaz/pruebas-esp32/main/10-control-lazo-cerrado/simulaciones/j1/j1_3_diseno_posicion.png" width="700"><br>
  <b>Fig. 11.</b>  Simulación del lazo de posición de J1 (base) con escalones de 20° y 5°: llega en 1,64 s sin sobrepico (Kpp = 2, v<sub>max</sub> = 30 °/s).
</p>


**J3 (codo):** escalones de 10 ° y 20 °. Llega en 1,3 a 1,7 s sin sobrepico; el PWM al subir crece con el ángulo por el peso.

<p align="center">
  <img src="https://raw.githubusercontent.com/HecVelaz/pruebas-esp32/main/10-control-lazo-cerrado/simulaciones/j3/j3_4_diseno_posicion.png" width="700"><br>
  <b>Fig. 12.</b>  Simulación del lazo de posición de J3 (codo) con escalones de 10° y 20°: llega en 1,3–1,7 s sin sobrepico (Kpp = 2, v<sub>max</sub> = 25 °/s). Al subir, el PWM crece con el ángulo por el peso del antebrazo.
</p>

#### Simulación contra la placa

| | Llegada en la placa | Llegada simulada | Sobrepico (placa / sim.) |
| :--- | :--- | :--- | :--- |
| J1, escalones de ±20 ° | 1,71 a 1,77 s | 1,64 s | 0 / 0 |
| J3, sube de 0 a +10 ° | 1,30 s | 1,29 s | 0 / 0 |
| J3, baja de 0 a −10 ° | 1,92 s | 1,06 s | 0 / 0 |

<p align="center">
  <img src="https://raw.githubusercontent.com/HecVelaz/pruebas-esp32/main/10-control-lazo-cerrado/simulaciones/j1/j1_5_comparacion_posicion_sim_vs_real.png" width="700"><br>
  <b>Fig. 13.</b>  Posición de J1 (base), placa (rojo) contra simulación (azul), en 20 escalones de ±20°. La placa llega en 1,71–1,77 s y la simulación en 1,64 s; ninguna se pasa del objetivo.
</p>

<p align="center">
  <img src="https://raw.githubusercontent.com/HecVelaz/pruebas-esp32/main/10-control-lazo-cerrado/simulaciones/j3/j3_7_comparacion_posicion_sim_vs_real_final.png" width="700"><br>
  <b>Fig. 14.</b>   Posición de J3 (codo), placa (rojo) contra simulación (azul), en 12 escalones de ±10°. Subiendo coinciden (1,30 s contra 1,29 s); bajando hacia −10° la placa tarda más (1,92 s contra 1,06 s) porque el mecanismo se endurece.
</p>

En J1 la simulación acierta: la placa llega 0,1 s más tarde por el juego de la correa. En J3 coincide subiendo, pero bajando hacia −10 ° la placa tarda casi el doble porque ahí el mecanismo se endurece, y el modelo no lo tiene.

El código del simulador está en el repositorio: `10-control-lazo-cerrado/tools/diseno_velocidad.py`, `diseno_posicion.py`, `escalon_velocidad.py`, `escalon_posicion.py` e `identificar_j3.py`. **J2 queda pendiente:** todavía no tiene modelo con el brazo montado.

---
### **7 - Implementación**

#### Estructura del firmware

El firmware se desarrolla en **PlatformIO** (Arduino-ESP32) en el proyecto `10-control-lazo-cerrado`, reutilizando los drivers de la Entrega 5 (enlazados, no copiados) y agregando el control en cascada:

| Módulo | Función |
| :--- | :--- |
| `lib/BTS7960/` | Driver del IBT-2 (q1 y q3): PWM con signo, freno y rueda libre |
| `lib/L298N/` | Driver del L298N en modo freno (q2, primera versión) |
| `lib/EncoderPCNT/` | Encoder en cuadratura x4 por el contador de pulsos (PCNT) del ESP32 |
| `include/config.h` | Parámetros de cada articulación: ganancias, compensación de zona muerta y de peso, límites y protecciones |
| `include/pins.h` | Pines de cada articulación (los mismos del robot) |
| `src/main.cpp` | Comandos, lazo de velocidad (PI), lazo de posición (P), protecciones y envío de datos |

Todo corre en una **ESP32-WROOM-32D**. Se compila una articulación por vez, eligiendo el entorno: `pio run -e j1` (base), `-e j2` (hombro) o `-e j3` (codo). Cuando una articulación queda probada, sus parámetros pasan al firmware del robot (`recolector_de_frutas/firmware/brazo`).

#### Lazo de control

El lazo corre con un período fijo de **10 ms**, medido con `micros()`. En cada período el firmware:

1. Lee el conteo del encoder y calcula la velocidad con la diferencia de cuentas de los últimos 40 ms.
2. Cada 20 ms (uno de cada dos períodos), el **lazo de posición** calcula la velocidad pedida: $\omega_{ref} = K_{pp}(\theta_{ref} - \theta)$, limitada a $v_{max}$ y a 60 °/s², y 0 si ya llegó (±0,3 °).
3. El **lazo de velocidad** calcula el PWM: compensación de zona muerta (y de peso en q3) más el PI, con anti-windup.
4. Limita el PWM y lo aplica al driver.
5. Revisa las protecciones y envía por serie una línea con el tiempo, la referencia, la posición, la velocidad y el PWM.

#### Código del controlador

Lazo de velocidad (PI), simplificado de `pasoPI()` en `src/main.cpp`. Unidades: velocidad en °/s y PWM en %.

```cpp
float pasoPI(float wRef, float wMed, float th, float &integral) {
  if (wRef == 0.0f) { integral = 0.0f; return 0.0f; }      // velocidad pedida 0: frena
  const float e = wRef - wMed;                              // error de velocidad [°/s]
  const int signo = wRef > 0 ? 1 : -1;
  // compensación de zona muerta; en q3 subiendo, también del peso según el ángulo
  const float ff = signo > 0 ? ffMarchaPos + ffPendPos * th : -ffMarchaNeg;
  const float uLibre = ff + kpVel * e + integral;           // PWM sin limitar [%]
  const float u = constrain(uLibre, -DUTY_MAX_PCT, DUTY_MAX_PCT);
  if (u == uLibre || (e > 0) != (uLibre > 0))               // anti-windup
    integral += kiVel * 0.01f * e;                          // Ts = 10 ms
  return u;
}
```

Lazo de posición (P), de `cascada()`:

```cpp
if (k % 2 == 0) {                                     // cada 20 ms
  const float e = thRef - grados(c);                  // error de posición [°]
  // llegado: dentro de ±0,3° (y no se aleja más de 0,6°)
  const float v = fminf(fmaxf(fabsf(kpp * e), VMIN_POS), VMAX_POS);
  const float deseada = llegado ? 0.0f : (e > 0 ? v : -v);
  const float dMax = AMAX_POS * 0.02f;                // aceleración máxima
  wRef = deseada == 0.0f ? 0.0f : constrain(deseada, wRef - dMax, wRef + dMax);
}
const float u = pasoPI(wRef, wMed, grados(c), integral);   // cada 10 ms
aplicarDuty(u);
```

#### Ganancias utilizadas

Las mismas que en la simulación, con las mismas unidades:

| | $K_p$ [%/(°/s)] | $K_i$ [%/°] | $K_{pp}$ [1/s] | Zona muerta ($u_{ff}$) | $v_{max}$ | PWM máx. |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| q1 (base) | 0,549 | 4 | 2 | 17 % | 30 °/s | 40 % |
| q3 (codo) | 0,35 | 5 | 2 | $24 + 1{,}4\,\theta$ % subiendo, 9 % bajando | 25 °/s | 80 % |

#### Comandos por puerto serie

| Comando | Función |
| :--- | :--- |
| `z` | Fija el ángulo actual como cero (con la articulación en su marca). Sin `z` no se mueve |
| `a <grados>` | Va a un ángulo con el control de posición |
| `ep <A> [ms] [ciclos]` | Escalones de posición $+A$, 0, $-A$, 0 (envía CSV) |
| `ev <°/s> [ms] [ciclos]` | Escalones de velocidad (envía CSV) |
| `bc <desde> <hasta> [°/s]` | Barrido lento a velocidad constante, para medir el PWM que hace falta según el ángulo (CSV) |
| `kp`, `ki`, `kpp <valor>` | Cambian las ganancias sin recargar el programa; `g` las muestra |
| `ff <valores>` | Cambia la compensación de zona muerta y de peso |
| `m <grados> [%]` | Va a un ángulo con PWM fijo, sin control (pruebas) |
| `p <%> [ms]` | Pulso de PWM fijo (pruebas) |
| `c`, `e` | Muestran la posición y el encoder |
| `x` o Enter | Frena: rueda libre / corta cualquier prueba |

#### Protecciones

- **Límite de ángulo** por software (q1: −45 ° a +135 °; q3: −15 ° a +28 °).
- **PWM máximo** configurable (40 % en q1, 80 % en q3).
- **Sin cuentas del encoder:** corta si con PWM aplicado el encoder no cuenta durante 150 ms (encoder suelto o motor trabado).
- **Sentido contrario:** corta si se mueve al revés de lo pedido durante 100 ms (signos mal configurados).
- **Atasco:** corta si hay PWM alto y no se mueve durante 300 ms.
- **Cero obligatorio:** no se mueve hasta hacer `z`, y cualquier Enter frena.
- **Límite de corriente en la fuente:** 3 A con q1, 2 A con q2 y q3.

#### Captura y procesamiento de datos

Los datos se reciben por serie en la PC con scripts de Python (`tools/escalon_velocidad.py`, `tools/escalon_posicion.py`, `tools/barrido_carga.py`), que mandan el comando, guardan el **CSV** en `resultados/` y lo grafican contra la simulación. Cada CSV empieza con una línea con los parámetros de la prueba y termina con `# fin ok` o `# cortado <causa>`. Columnas:

- Escalones de posición: `t_ms, th_ref, th, w_ref, w_med, duty`
- Escalones de velocidad y barridos: `t_ms, w_ref, w_med, duty, pos`

---
### **8 - Validación**

La validación compara la respuesta **medida** en el brazo con la **simulada**, para la misma referencia, las mismas ganancias y la misma compensación. Se validaron J1 (base) y J3 (codo), montados en el brazo.

#### Ensayos

| Ensayo | Descripción | J1 (base) | J3 (codo) |
| :--- | :--- | :--- | :--- |
| Escalones de velocidad | $+v$, 0, $-v$, 0 repetido | ±20 °/s, 80 escalones | ±10 °/s, 6 escalones ($K_p$ = 0,5) |
| Escalones de posición | 0 → $+A$ → 0 → $-A$ → 0 repetido | ±20 °, 20 escalones | ±10 °, 2 pruebas de 12 escalones |
| Con carga | Escalones de posición con un limón en la pinza | ±20 °, 20 escalones | — |
| Perturbación | Frenar la base con la mano mientras se mueve | escalón de 15 °/s | — |
| Repetibilidad | Dispersión entre repeticiones del mismo escalón | 5 por tipo de escalón | 3 por tipo, en 2 pruebas |
| Verificación del ángulo | Medir el ángulo con un instrumento externo | doblez de 45 ° en una hoja | inclinómetro |

Las ganancias no se modificaron durante ningún ensayo. Todos se hicieron con las ganancias finales de cada articulación, salvo los escalones de velocidad de J3 y su primera prueba de posición, que usaron $K_p$ = 0,5 en lugar de 0,35.

#### Métricas

- **Sobrepico:** $M_p = \dfrac{\theta_{max} - \theta_{ref}}{\theta_{ref}} \times 100\,\%$
- **Tiempo de llegada:** $t_s$ hasta entrar y permanecer a menos de **0,5 °** de la referencia. No se usa la banda del 2 % porque el control frena al llegar a ±0,3 ° del objetivo: con un escalón de 10 °, la banda del 2 % (±0,2 °) es más chica que esa tolerancia.
- **Error en régimen permanente:** $e_{ss} = \theta_{ref} - \theta_{final}$.
- **RMSE** entre la simulación y la medición, en el primer ciclo de cada prueba: $\sqrt{\dfrac{1}{N}\sum_k \left(\theta_{sim}[k] - \theta_{med}[k]\right)^2}$.

#### Resultados

| Motor | Ensayo | $M_p$ [%] | $t_s$ [s] | $e_{ss}$ [°] | RMSE [°] |
| :--- | :--- | :--- | :--- | :--- | :--- |
| J1 | Posición ±20 ° (20 escalones) | 0 | 1,71 – 1,77 | 0,27 – 0,30 | 0,49 |
| J1 | Posición ±20 ° con carga (20 escalones) | 0 | 1,67 – 1,75 | 0,22 – 0,32 | — |
| J1 | Velocidad ±20 °/s (80 escalones) | 3,9 – 7,1 | 0,33 – 0,38 | 0,14 – 0,54 °/s | — |
| J3 | Posición ±10 °, primera versión (12 escalones) | 0 – 1,4 | 1,30 – 1,67 | ≤ 0,28 | 0,45 |
| J3 | Posición ±10 °, versión final (2 × 12 escalones) | 0 | 0,96 – 1,92 | ≤ 0,29 | 1,07 – 1,26 |

**Repetibilidad:** entre repeticiones del mismo escalón, el tiempo de llegada varía ±0,01 s en J1 y ±0,01 a 0,02 s en J3.

**Perturbación (J1):** al frenar la base con la mano a 15 °/s, la velocidad bajó a 11,8 °/s y el PI subió el PWM de 21,7 % a 25,8 %. Al soltarla, la base volvió a 15 °/s en unos 0,8 s (figura 16). Quieta, la base no se puede mover a mano: la reducción total es de 250:1.

**Verificación del ángulo:** en J1, la orden de ir a 45 ° quedó paralela a un doblez de 45 ° en una hoja (verificado a ojo). En J3, el encoder marcó 12,6 ° y el inclinómetro 13 °: el factor de 34,57 cuentas por grado del codo es correcto con una diferencia de unos 0,5 °.

Las figuras 13 y 14 muestran la simulación superpuesta a la medición de la posición de J1 y J3. Las figuras 15 y 16 muestran la velocidad de J1, y las figuras 17 y 18, la velocidad y la primera prueba de posición de J3.

<p align="center">
  <img src="https://raw.githubusercontent.com/HecVelaz/pruebas-esp32/main/10-control-lazo-cerrado/simulaciones/j1/j1_4_comparacion_velocidad_sim_vs_real.png" width="750"><br>
  <b>Fig. 15.</b>  Velocidad de J1 (base), placa contra simulación: 80 escalones de ±20 °/s superpuestos (rojo claro), su promedio (rojo) y la simulación (azul). La placa llega a la velocidad pedida un poco antes y se pasa 4–7 %, contra 1 % en la simulación.
</p>

<p align="center">
  <img src="https://raw.githubusercontent.com/HecVelaz/pruebas-esp32/main/10-control-lazo-cerrado/imagenes/j1_base/j1_15_escalon_velocidad_20261002_225238.png" width="750"><br>
  <b>Fig. 16.</b>  Perturbación en J1: escalón de velocidad de 15 °/s con la base frenada a mano entre 1,2 y 1,4 s. La velocidad cae a 11,8 °/s, el PI aumenta el PWM y la base vuelve a 15 °/s en unos 0,8 s.
</p>

<p align="center">
  <img src="https://raw.githubusercontent.com/HecVelaz/pruebas-esp32/main/10-control-lazo-cerrado/simulaciones/j3/j3_5_comparacion_velocidad_sim_vs_real.png" width="750"><br>
  <b>Fig. 17.</b>  Velocidad de J3 (codo), placa contra simulación: 3 escalones de ±10 °/s superpuestos (Kp = 0,5, Ki = 5). Bajando coinciden; subiendo, la placa tiene un pico de ~20 °/s al arrancar por el juego del piñón y el engranaje, que el modelo no tiene.
</p>

<p align="center">
  <img src="https://raw.githubusercontent.com/HecVelaz/pruebas-esp32/main/10-control-lazo-cerrado/simulaciones/j3/j3_6_comparacion_posicion_sim_vs_real_kp05.png" width="700"><br>
  <b>Fig. 18.</b>  Posición de J3 (codo), primera versión (Kp = 0,5, Ki = 5, v<sub>max</sub> = 15 °/s), placa (rojo) contra simulación (azul), en 12 escalones de ±10°. Las llegadas coinciden (1,60 / 1,30 / 1,35 s contra 1,60 / 1,31 / 1,36 s), salvo bajando de 0 a −10° (1,67 s contra 1,32 s).
</p>

#### Discusión

**Cumplimiento de las especificaciones de la sección 6:** en J1 y J3 la posición se cumple en todos los ensayos: sin sobrepico, llegada en menos de 2 s y error final por debajo de 0,5 °. El lazo de velocidad de J1 quedó en $t_s$ = 0,33 – 0,38 s y sobrepico de 4 – 7 %. Es más lento que los 0,25 s del diseño, porque $K_i$ se bajó de 8,44 a 4 para reducir el sobrepico que causaba el juego de la correa.

**Diferencias entre simulación y medición:**

- **J1:** la placa llega unos 0,1 s más tarde que la simulación (RMSE de 0,49 °). La causa es el juego de la correa: el encoder está en el motor, y al invertir el giro el motor recorre el juego antes de mover la base. El error final real (0,27 – 0,30 °) es mayor que el simulado (0,05 – 0,09 °), porque en la placa la fricción y el freno la detienen apenas entra en la tolerancia de ±0,3 °.
- **J3:** subiendo coinciden (1,30 s contra 1,29 s). Bajando hacia −10 ° la placa tarda casi el doble (1,92 s contra 1,06 s): el mecanismo de cuatro barras se endurece cerca de su posición más baja, y el modelo supone una fricción constante al bajar. Por eso el RMSE de J3 (1,1 – 1,3 °) es mayor que el de J1. También hay picos de velocidad al empezar a subir, por el juego del piñón y el engranaje, y un "traba-suelta" del tornillo sin fin al bajar despacio.
- **Carga:** el limón no cambia la respuesta de J1, porque con la reducción de 250:1 su inercia casi no se nota en el motor.

**Limitaciones:**

- J1 usa el modelo del motor con el eje libre, que alcanzó para predecir la respuesta con el brazo montado. J3 se re-identificó con el brazo montado (sección 5). J2 todavía no tiene modelo con carga ni validación.
- Las verificaciones del ángulo fueron simples (un doblez de papel y un nivelador). Falta una verificación con transportador o con escala en video.
- No se hizo el ensayo de perturbación en J3.

---
### **9 - Videos demostrativos de la simulación y de las implementaciones**


**Demostrador, motor base J1 (36GP-555):**  Se muestra el control en lazo cerrado de la base del brazo.

[![Ver video](https://img.youtube.com/vi/VCHOl4hgZ3g/hqdefault.jpg)](https://www.youtube.com/watch?v=VCHOl4hgZ3g&list=PLQCLakFxinI0&index=2)

- [Prueba en lazo cerrado de la articulación J1](https://www.youtube.com/watch?v=VCHOl4hgZ3g&list=PLQCLakFxinI0&index=2)

**Demostrador, articulación J3 (5840-31ZY):** Se muestra el control en lazo cerrado, el brazo pasa de 0º a 10º y de 0º a -10º.


[![Ver video](https://img.youtube.com/vi/rtzFyMSvHHg/hqdefault.jpg)](https://www.youtube.com/watch?v=rtzFyMSvHHg&list=PLQCLakFxinI0&index=1)

- [Prueba en lazo cerrado de la articulación J3](https://www.youtube.com/watch?v=rtzFyMSvHHg&list=PLQCLakFxinI0&index=1)

### **10 - Referencias**

[1] Infineon Technologies, *BTS7960 High Current PN Half Bridge NovalithIC™*, hoja de datos.

[2] STMicroelectronics, *L298 Dual Full-Bridge Driver*, hoja de datos.

[3] Espressif Systems, ejemplo *rotary_encoder* de ESP-IDF v4.4. Disponible en: <https://github.com/espressif/esp-idf/tree/v4.4/examples/peripherals/pcnt/rotary_encoder>.

[4] PlatformIO, *Espressif 32 — PlatformIO documentation*. Disponible en: <https://docs.platformio.org/en/latest/platforms/espressif32.html>.

[5] Naylamp Mechatronics, “Ejemplos_L298N”, GitHub. Disponible en: <https://github.com/naylampmechatronics/Ejemplos_L298N>.


---
### **Carpeta del proyecto**
En el link se encuentra la carpeta del proyecto.
[TPF R1 Grupo 2](https://drive.google.com/drive/folders/1d2jONLnkB8FMkP2iqF4ia4UeQ2jRpffz)
